In [14]:
import itertools
import numpy as np
import pandas as pd

# Set seed for reproducibility
np.random.seed(42)

# 20 Official Italian Regions
regions = [
    "Abruzzo", "Basilicata", "Calabria", "Campania", "Emilia-Romagna",
    "Friuli-Venezia Giulia", "Lazio", "Liguria", "Lombardy", "Marche",
    "Molise", "Piedmont", "Apulia", "Sardinia", "Sicily",
    "Tuscany", "Trentino-Alto Adige", "Umbria", "Aosta Valley", "Veneto"
]

years = range(2011, 2026)

# Main commercial and premium Italian wine segments
wine_types = ['Sangiovese (Red)', 'Nebbiolo (Red)', 'Glera (Prosecco/Sparkling)', 'Pinot Grigio (White)', 'Primitivo (Red)', 'Montepulciano (Red)']

# Regional population for metadata assignment
region_populations = {
    "Lombardy": 10000000, "Lazio": 5700000, "Campania": 5600000, "Veneto": 4800000,
    "Sicily": 4800000, "Emilia-Romagna": 4400000, "Piedmont": 4200000, "Apulia": 3900000,
    "Tuscany": 3600000, "Calabria": 1800000, "Sardinia": 1600000, "Liguria": 1500000,
    "Marche": 1500000, "Abruzzo": 1200000, "Friuli-Venezia Giulia": 1200000,
    "Trentino-Alto Adige": 1100000, "Umbria": 850000, "Basilicata": 540000,
    "Molise": 290000, "Aosta Valley": 123000
}

# 1. Define typical yields (Hectoliters per Hectare - hl/ha)
# Premium reds have low yields; sparkling/whites have high mass production yields
wine_yield_profiles = {
    'Sangiovese (Red)': (65.0, 8.0),       # Standard Tuscan / Central yield
    'Nebbiolo (Red)': (48.0, 5.0),         # Strict, highly premium low yield (Barolo/Barbaresco)
    'Glera (Prosecco/Sparkling)': (110.0, 12.0), # Very high legal yields allowed for Prosecco
    'Pinot Grigio (White)': (90.0, 10.0),   # Higher yielding commercial white
    'Primitivo (Red)': (75.0, 9.0),        # Robust southern red yield
    'Montepulciano (Red)': (85.0, 10.0)    # Generous yield high-volume red
}

# 2. Define Regional Scale Factors for Hectares (Who has the most vineyard footprints?)
# Veneto, Apulia, Sicily, and Emilia-Romagna dominate bulk and scale infrastructure.
region_scale = {
    "Veneto": 1.6, "Apulia": 1.5, "Sicily": 1.4, "Emilia-Romagna": 1.3,
    "Tuscany": 1.1, "Piedmont": 1.1, "Abruzzo": 0.9, "Friuli-Venezia Giulia": 0.8,
    "Campania": 0.6, "Lombardy": 0.6, "Lazio": 0.5, "Marche": 0.5,
    "Sardinia": 0.4, "Umbria": 0.3, "Trentino-Alto Adige": 0.3, "Calabria": 0.2,
    "Basilicata": 0.1, "Molise": 0.1, "Liguria": 0.05, "Aosta Valley": 0.02 # Tiny mountain alpine plots
}

# 3. Define Wine-to-Region Suitability Matrix
# If a wine isn't traditionally grown in a region, its footprint is heavily compressed
suitability_matrix = {wine: {reg: 0.1 for reg in regions} for wine in wine_types}

# Tuscany & Central Italy love Sangiovese
for r in ["Tuscany", "Emilia-Romagna", "Umbria", "Marche", "Lazio"]:
    suitability_matrix['Sangiovese (Red)'][r] = 1.5

# Piedmont owns Nebbiolo (almost exclusively grown there)
suitability_matrix['Nebbiolo (Red)']['Piedmont'] = 1.8
suitability_matrix['Nebbiolo (Red)']['Lombardy'] = 0.4 # Tiny bit in Valtellina

# Veneto and Northeast dominate Glera (Prosecco) and Pinot Grigio
for r in ["Veneto", "Friuli-Venezia Giulia", "Trentino-Alto Adige"]:
    suitability_matrix['Glera (Prosecco/Sparkling)'][r] = 1.9
    suitability_matrix['Pinot Grigio (White)'][r] = 1.6

# Deep South loves Primitivo
for r in ["Apulia", "Basilicata", "Calabria"]:
    suitability_matrix['Primitivo (Red)'][r] = 1.7

# Adriatic Coast loves Montepulciano
for r in ["Abruzzo", "Marche", "Molise", "Apulia"]:
    suitability_matrix['Montepulciano (Red)'][r] = 1.5

# Base vineyard size baseline for an individual wine type allocation
BASE_AREA_LOC = 4000 
BASE_AREA_SCALE = 1000

combinations = list(itertools.product(regions, years, wine_types))

rows = []
for region, year, wine in combinations:
    r_scale = region_scale.get(region, 1.0)
    w_suit = suitability_matrix[wine].get(region, 0.1)
    
    # Generate unique Hectares base
    area_loc = BASE_AREA_LOC * r_scale * w_suit
    area_scale = BASE_AREA_SCALE * r_scale * w_suit
    
    # Ensure even tiny alpine regions keep a realistic baseline footprint if suitable
    hectares = max(5, np.random.normal(loc=area_loc, scale=area_scale))
    
    # Slight systemic economic/market changes year-by-year (e.g. Prosecco boom post-2015)
    market_trend = 1.0
    if wine == 'Glera (Prosecco/Sparkling)' and year > 2015:
        market_trend = 1.35
        
    hectares = int(hectares * market_trend * np.random.uniform(0.9, 1.1))
    
    # Calculate Yield (Hectoliters per Hectare)
    yield_mean, yield_std = wine_yield_profiles[wine]
    
    # Introduce systemic bad weather years in Italy (e.g., 2017 hit by extreme spring frost/drought)
    if year == 2017:
        yield_mean *= 0.86
    # 2023 was famously hit hard by downy mildew disease across central/southern Italy
    elif year == 2023:
        yield_mean *= 0.80
        
    actual_yield = max(15.0, np.random.normal(loc=yield_mean, scale=yield_std))
    
    # Production calculation
    production_hl = int(hectares * actual_yield)
    
    rows.append({
        'Region': region,
        'Year': year,
        'WineType': wine,
        'Production_Hectoliters': production_hl,
        'Area_Hectares': hectares
    })

df = pd.DataFrame(rows)
df['Population'] = df['Region'].map(region_populations)

region_groups = {
    "Abruzzo": "Center",
    "Basilicata": "South",
    "Calabria": "South",
    "Campania": "South",
    "Emilia-Romagna": "North",
    "Friuli-Venezia Giulia": "North",
    "Lazio": "Center",
    "Liguria": "North",
    "Lombardia": "North",
    "Marche": "Center",
    "Molise": "South",
    "Piemonte": "North",
    "Puglia": "South",
    "Sardegna": "Islands",
    "Sicilia": "Islands",
    "Toscana": "Center",
    "Trentino-Alto Adige": "North",
    "Umbria": "Center",
    "Valle D'Aosta": "North",
    "Veneto": "North"
}

df['Area'] = df['Region'].map(region_groups)

# Helper metrics check for students
df['Yield_Hl_Ha'] = (df['Production_Hectoliters'] / df['Area_Hectares']).round(1)

print(df.sample(10))

              Region  Year                    WineType  \
284         Campania  2013  Glera (Prosecco/Sparkling)   
876           Marche  2022            Sangiovese (Red)   
575            Lazio  2016         Montepulciano (Red)   
732         Lombardy  2013            Sangiovese (Red)   
1289          Sicily  2015         Montepulciano (Red)   
964           Molise  2021             Primitivo (Red)   
587            Lazio  2018         Montepulciano (Red)   
1291          Sicily  2016              Nebbiolo (Red)   
969           Molise  2022        Pinot Grigio (White)   
426   Emilia-Romagna  2022            Sangiovese (Red)   

      Production_Hectoliters  Area_Hectares  Population    Area  Yield_Hl_Ha  
284                    25838            220     5600000   South        117.4  
876                   174214           2739     1500000  Center         63.6  
575                    12113            148     5700000  Center         81.8  
732                    15863            236  

In [15]:
# Ensure non-negative values for production and area
# Забезпечення невід'ємних значень для виробництва та площі
df['Production_Hectoliters'] = df['Production_Hectoliters'].apply(lambda x: max(0, x))
df['Area_Hectares'] = df['Area_Hectares'].apply(lambda x: max(1, x)) # Area should be at least 1

In [16]:
# --- Introduce intentional data issues for cleaning exercises ---
# --- Введення навмисних проблем з даними для вправ з очищення ---

n_rows = len(df)

# 1. Missing values (approx 5-10%)
# 1. Пропущені значення (приблизно 5-10%)
for col in ['Production_Hectoliters', 'Area_Hectares', 'Yield_Hl_Ha']:
    missing_indices = np.random.choice(df.index, int(n_rows * np.random.uniform(0.05, 0.1)), replace=False)
    df.loc[missing_indices, col] = np.nan

# 2. Outliers: introduce some extremely high values
# 2. Викиди: введення деяких надзвичайно високих значень
outlier_indices = np.random.choice(df.index, int(n_rows * 0.025), replace=False)
df.loc[outlier_indices, 'Production_Hectoliters'] = df.loc[outlier_indices, 'Production_Hectoliters'] * np.random.uniform(5, 10)

# 3. Inconsistent string data
# 3. Непослідовні текстові дані
#   - Inconsistent capitalization for CropType
#   - Непослідовна велика/мала літера для CropType
lower_case_indices = np.random.choice(df.index, int(n_rows * 0.05), replace=False)
df.loc[lower_case_indices, 'WineType'] = df.loc[lower_case_indices, 'WineType'].str.lower()

#   - Variations in Region naming (e.g., 'Kyiv region', 'Lviv Reg.')
#   - Варіації у назвах регіонів (наприклад, 'Київський регіон', 'Львівська обл.')
region_variations_indices = np.random.choice(df.index, int(n_rows * 0.05), replace=False)
df.loc[region_variations_indices, 'Region'] = df.loc[region_variations_indices, 'Region'].apply(lambda x: x.lower())

# 4. Incorrect data types (e.g., some numeric columns might be objects if mixed with text in real data)
#    For this synthetic data, we ensure they are numeric, but this is a common real-world issue.
# 4. Неправильні типи даних (наприклад, деякі числові стовпці можуть бути об'єктами, якщо змішані з текстом у реальних даних).
#    Для цих синтетичних даних ми гарантуємо, що вони є числовими, але це поширена проблема в реальному світі.

print("Original Data Sample (with intentional issues):")
print("Приклад оригінальних даних (з навмисними проблемами):")

Original Data Sample (with intentional issues):
Приклад оригінальних даних (з навмисними проблемами):


In [17]:
df.to_csv('data_workshop_project.csv')
df

,Region,Year,WineType,Production_Hectoliters,Area_Hectares,Population,Area,Yield_Hl_Ha
0,Abruzzo,2011,Sangiovese (Red),27027.0,423.0,1200000,Center,63.9
1,Abruzzo,2011,Nebbiolo (Red),12001.0,242.0,1200000,Center,49.6
2,Abruzzo,2011,Glera (Prosecco/Sparkling),NaN,453.0,1200000,Center,119.2
3,abruzzo,2011,Pinot Grigio (White),24407.0,288.0,1200000,Center,84.7
4,Abruzzo,2011,Primitivo (Red),21725.0,376.0,1200000,Center,57.8
...,...,...,...,...,...,...,...,...
1795,Veneto,2025,Nebbiolo (Red),44873.0,849.0,4800000,North,52.9
1796,Veneto,2025,Glera (Prosecco/Sparkling),NaN,17363.0,4800000,North,138.7
1797,Veneto,2025,Pinot Grigio (White),702381.0,8004.0,4800000,North,87.8
1798,Veneto,2025,Primitivo (Red),49422.0,617.0,4800000,North,80.1
